In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from langdetect import detect, DetectorFactory

In [2]:
# Cố định seed cho langdetect để kết quả luôn nhất quán mỗi lần chạy
DetectorFactory.seed = 0

# Tên file dữ liệu của bạn
CAPTION_FILE = r'C:\Users\Administrator\Desktop\New folder\dataset\captions_en-vi.csv'

In [8]:
# Đọc file, bỏ qua các dòng lỗi format
df = pd.read_csv(CAPTION_FILE, on_bad_lines='skip', encoding='utf-8')

print(f"Tổng số dòng dữ liệu đọc được ban đầu: {len(df)}")

# Kiểm tra missing values
missing_values = df.isnull().sum()
print(f"Số dòng bị thiếu tên ảnh (null image): {missing_values.get('image', 0)}")
print(f"Số dòng bị thiếu caption (null caption): {missing_values.get('caption', 0)}")

# Loại bỏ các dòng bị thiếu dữ liệu để phân tích cho chuẩn xác
df = df.dropna(subset=['image', 'caption']).copy()
print(f"Số dòng sau khi xóa giá trị thiếu: {len(df)}")

# Xem thử 30 dòng đầu tiên
df.head(10)


Tổng số dòng dữ liệu đọc được ban đầu: 80308
Số dòng bị thiếu tên ảnh (null image): 0
Số dòng bị thiếu caption (null caption): 0
Số dòng sau khi xóa giá trị thiếu: 80308


,image,caption
0,1009434119_febe49276a.jpg,A black and white dog is running in a grassy g...
1,1009434119_febe49276a.jpg,A black and white dog is running through the g...
2,1009434119_febe49276a.jpg,A Boston terrier is running in the grass .
3,1009434119_febe49276a.jpg,A Boston Terrier is running on lush green gras...
4,1009434119_febe49276a.jpg,A dog runs on the green grass near a wooden fe...
5,1012212859_01547e3f17.jpg,"A dog shakes its head near the shore , a red b..."
6,1012212859_01547e3f17.jpg,A white dog shakes on the edge of a beach with...
7,1012212859_01547e3f17.jpg,"Dog with orange ball at feet , stands on shore..."
8,1012212859_01547e3f17.jpg,White dog playing with a red ball on the shore...
9,1012212859_01547e3f17.jpg,White dog with brown ears standing near water ...


In [7]:
# # xem 30 dòng cuối
df.tail(10)

,image,caption
80298,979201222_75b6456d34.jpg,Hai cô gái đang nắm tay trong bàn tay trong qu...
80299,979201222_75b6456d34.jpg,Hai cô gái đang rời khỏi camera dưới một lối đ...
80300,979201222_75b6456d34.jpg,Hai cô gái mặc quần đùi đi đến những biển báo
80301,979201222_75b6456d34.jpg,có hai cô gái đi dạo trong một cơn gió nhẹ
80302,979201222_75b6456d34.jpg,có hai người đàn bà đi bộ trên vỉa hè
80303,99171998_7cc800ceef.jpg,Một nhóm đang ngồi xung quanh một khe nứt tuyết
80304,99171998_7cc800ceef.jpg,Một nhóm người ngồi trên đỉnh núi tuyết
80305,99171998_7cc800ceef.jpg,Một nhóm người ngồi trong tuyết rơi trong một ...
80306,99171998_7cc800ceef.jpg,5 đứa sẵn sàng để bắt đầu
80307,99171998_7cc800ceef.jpg,Năm người đang ngồi bên nhau trong tuyết


In [ ]:
import re

#  Xóa dấu ngoặc kép thừa và sửa lỗi khoảng trắng trước dấu câu
def clean_text(text):
    text = str(text).strip('"\'') 
    text = re.sub(r'\s+([,.?!])', r'\1', text) # VD: "ball ." -> "ball."
    text = re.sub(r'\s+', ' ', text) # Gộp nhiều khoảng trắng thành 1
    return text.strip()

In [ ]:
#  Xử lý lỗi caption bị lặp lại 2 nửa giống hệt nhau
def fix_repeats(text):
    words = text.split()
    n = len(words)
    
    # Xử lý lặp nguyên câu (VD: "Một con chó Một con chó")
    if n > 2 and n % 2 == 0:
        half = n // 2
        if words[:half] == words[half:]:
            return " ".join(words[:half])
            
    # Xử lý lặp 1 từ liên tục ở cuối (VD: "dương dương dương dương...")
    # Bằng cách nén các từ giống hệt nhau đứng liền kề thành 1 từ
    cleaned_words = []
    for word in words:
        if not cleaned_words or word != cleaned_words[-1]:
            cleaned_words.append(word)
            
    return " ".join(cleaned_words)

In [11]:
print("Đang dọn dẹp văn bản (xóa nhiễu, sửa lỗi lặp)...")

# Lưu lại caption gốc để  so sánh
df['original_caption'] = df['caption']

# Áp dụng các hàm làm sạch
df['caption'] = df['caption'].apply(clean_text)
df['caption'] = df['caption'].apply(fix_repeats)

# Xem thử những dòng đã được sửa đổi (để kiểm tra xem hàm chạy đúng không)
changed_df = df[df['original_caption'] != df['caption']]
print(f"Đã phát hiện và sửa lỗi cho {len(changed_df)} dòng.")

# Hiển thị thử 5 dòng đã sửa
changed_df[['original_caption', 'caption']].head(5)

Đang dọn dẹp văn bản (xóa nhiễu, sửa lỗi lặp)...
Đã phát hiện và sửa lỗi cho 38337 dòng.


,original_caption,caption
0,A black and white dog is running in a grassy g...,A black and white dog is running in a grassy g...
1,A black and white dog is running through the g...,A black and white dog is running through the g...
2,A Boston terrier is running in the grass .,A Boston terrier is running in the grass.
3,A Boston Terrier is running on lush green gras...,A Boston Terrier is running on lush green gras...
4,A dog runs on the green grass near a wooden fe...,A dog runs on the green grass near a wooden fe...


In [12]:
# Đếm số lượng từ sau khi đã làm sạch
df['word_count'] = df['caption'].apply(lambda x: len(x.split()))

# Chỉ giữ lại các caption có từ 2 chữ trở lên
final_df = df[(df['word_count'] >= 5) & (df['word_count'] <= 75)].copy()

# Báo cáo
removed = len(df) - len(final_df)
print(f"Số dòng rác/quá ngắn bị loại bỏ: {removed}")
print(f"Số dòng CỰC SẠCH còn lại để train: {len(final_df)}")

# Cấu trúc lại: CHỈ GIỮ LẠI ĐÚNG 2 CỘT (image và caption)
final_df = final_df[['image', 'caption']]
final_df.head()

Số dòng rác/quá ngắn bị loại bỏ: 789
Số dòng CỰC SẠCH còn lại để train: 79519


,image,caption
0,1009434119_febe49276a.jpg,A black and white dog is running in a grassy g...
1,1009434119_febe49276a.jpg,A black and white dog is running through the g...
2,1009434119_febe49276a.jpg,A Boston terrier is running in the grass.
3,1009434119_febe49276a.jpg,A Boston Terrier is running on lush green gras...
4,1009434119_febe49276a.jpg,A dog runs on the green grass near a wooden fe...


In [19]:
# Đường dẫn lưu file đầu ra (Lưu cùng thư mục với file gốc)
import os

input_dir = os.path.dirname(CAPTION_FILE)
OUTPUT_FILE = os.path.join(input_dir, r'C:\Users\Administrator\Desktop\New folder\dataset\captions_ready_to_train.csv')

print(f"Đang lưu file kết quả vào: {OUTPUT_FILE}")
final_df.to_csv(OUTPUT_FILE, index=False, encoding='utf-8')

print("File  đã lưu.")

Đang lưu file kết quả vào: C:\Users\Administrator\Desktop\New folder\dataset\captions_ready_to_train.csv
File  đã lưu.


In [27]:
import os

# 1. Thiết lập đường dẫn thư mục đầu ra
OUTPUT_DIR = r'C:\Users\Administrator\Desktop\New folder\dataset\train'

# Khởi tạo thư mục nếu chưa tồn tại
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("[INFO] Bắt đầu quá trình trích xuất dữ liệu...")


csv_path = os.path.join(OUTPUT_DIR, 'captions_ready_to_train.csv')
final_df.to_csv(csv_path, index=False, encoding='utf-8')
print(f"[SUCCESS] Đã lưu tệp CSV tại: {csv_path}")


jsonl_path = os.path.join(OUTPUT_DIR, 'metadata.jsonl')

# Đồng bộ hóa trường dữ liệu theo tiêu chuẩn Text-to-Image
hf_df = final_df.rename(columns={'image': 'file_name', 'caption': 'text'})

# Trích xuất JSON Lines, giữ nguyên bảng mã UTF-8
hf_df.to_json(jsonl_path, orient='records', lines=True, force_ascii=False)
print(f"[SUCCESS] Đã lưu tệp JSONL tại: {jsonl_path}")

print("\n[ĐÃ HOÀN TẤT] Dữ liệu sẵn sàng cho quá trình huấn luyện mô hình.")

[INFO] Bắt đầu quá trình trích xuất dữ liệu...
[SUCCESS] Đã lưu tệp CSV tại: C:\Users\Administrator\Desktop\New folder\dataset\train\captions_ready_to_train.csv
[SUCCESS] Đã lưu tệp JSONL tại: C:\Users\Administrator\Desktop\New folder\dataset\train\metadata.jsonl

[ĐÃ HOÀN TẤT] Dữ liệu sẵn sàng cho quá trình huấn luyện mô hình.
